In [0]:
import requests
import json
from datetime import datetime, timezone, timedelta
from pyspark.sql.functions import col, parse_json
import uuid

In [0]:
# Define destination path inside Unity Catalog Volume
write_path = "the_data_masons.landing"

In [0]:
# Set API url
url = 'https://api.eia.gov/v2/electricity/rto/region-sub-ba-data/data/'
api_key = "52QFUWuo1qHOtc3jnd40P6cTGznr1fLmpltudeq6"
period_start_date = "2026-01-01T00"
time_period = "2026-01-02T00"
LENGTH = 5000

# Define file prefix
table_name = "raw_demand_by_zone"

In [0]:
def build_Xparams(tracker):
    print("Building Xparams")
    return {
        "frequency":"hourly",
        "data": ["value"],
        "start":tracker['target_dt'],
        "end":tracker['target_dt'],
        "facets": {
            "parent": [
                "NYIS"
            ]
        },
        "sort": [
            {
                "column": "period",
                "direction": "asc"
            },
            {
                "column": "parent",
                "direction": "asc"
            },
            {
                "column": "subba",
                "direction": "asc"
            }
        ],
        "offset": tracker['offset'],
        "length": 5000
    }
    

In [0]:
def fetch_response(url, params, headers):
    print("Fetching data")
    return requests.get(url, params= params, headers= headers)

In [0]:
def parse_response(response):
    print("Parsing response")

    # check response status
    response.raise_for_status()

    return response.json()

In [0]:
def construct_record(tracker, response):
    print("Constructing record")

    record_buffer = {
        'run_id' : tracker['run_id'],
        "request_timestamp": tracker['timestamp'],
        'target_dt':tracker['target_dt'],
        'page': tracker['page'],
        'offset_start':tracker['offset'],
        "status_code": response.status_code,
        "status": response.reason
    }

    if record_buffer["status_code"] == 200:
        print("Status code is 200")
        data = response.json()
        raw_response = json.dumps(data)
        records = data.get("response",{}).get("data",[])
        record_count = len(records)
        error_msg = 'None'
    else:
        print(f"Status code is {record_buffer["status_code"]}")
        error_msg = response.text if hasattr(response, 'text') else str(response.get("error"))
        raw_response = "{}"
        record_count = 0

    record_buffer.update({
        "raw_response": raw_response,
        "error_msg": error_msg,
        "record_count": record_count
    })

    return record_buffer

In [0]:
def create_dataframe(buffer):
    print("Creating dataframe from buffer")
    return (spark.createDataFrame(buffer)
    .withColumn("raw_response", parse_json(col("raw_response"))))

In [0]:
def target_hours(period_start_date):
    print("Calculating target hours")
    start_dt = datetime.strptime(period_start_date, "%Y-%m-%dT%H").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(time_period, "%Y-%m-%dT%H").replace(tzinfo=timezone.utc)

    hours = []
    curr = start_dt
    while curr <= end_dt:
        hours.append(curr.strftime("%Y-%m-%dT%H"))
        curr += timedelta(hours=1)

    return hours
    # print(f"Target hours: {hours}")

In [0]:
if __name__ == '__main__':
# def main():
    # create a Delta Table buffer array
    landing_buffer = []
    tracker = {
        "run_id" : "",
        'timestamp': datetime.now(timezone.utc),
        'target_dt' : '',
        "offset" : 0,
        "page" : 0
    }

    dt_list = target_hours(period_start_date)

    for hour in dt_list:

        # generate run id
        tracker["run_id"] = str(uuid.uuid4())
        tracker['offset'] = 0
        tracker['target_dt'] = hour
        has_more_data = True

        # loop and append response record to the buffer
        while has_more_data:

            # set page number
            tracker["page"] = (tracker["offset"] // LENGTH) + 1

            # build request params
            headers = {"X-Params" : json.dumps(build_Xparams(tracker))}
            params = {"api_key": api_key}

            # get request time
            tracker['timestamp'] = datetime.now(timezone.utc)

            # get request resonse
            # fetch_response() returns a response object
            response = fetch_response(url, params, headers)

            # check for errors
            response.raise_for_status()

            # no errors, then Construct record
            # construct_record() returns 
            landing_record = construct_record(tracker, response)

            # update tracker
            if landing_record["status_code"] != 200:
                print(f"Failed to fetch data, status code: {landing_record["status_code"]}")
                has_more_data = False
            elif landing_record["record_count"] < LENGTH:
                has_more_data = False
            else:
                tracker["offset"] += LENGTH
            
            

            # append record to buffer
            print(f"Appending record {i} to buffer")
            print(landing_record)   
            landing_buffer.append(landing_record)
        
    # create a dataframe from the buffer array
    df_landing = create_dataframe(landing_buffer)
                                                
    display(df_landing)


In [0]:
# if __name__ == '__main__':
#     main()

In [0]:
# # create a dataframe from the buffer array
# df_landing = create_dataframe(landing_buffer)
                                              
# display(df_landing)

In [0]:
# # Save the table to the landing schema
# df_landing.write.mode("append").saveAsTable(f"{write_path}.{table_name}")   